# Notebook 2 - Dataset Repair
### Matched negatives, PyPI expansion, and a working AST feature

Notebook 1 showed three problems that no amount of rewriting can fix:

| Problem | Evidence from Notebook 1 |
|---|---|
| benign class confounded with time/popularity/size | ecosystem-only baseline F1 = 0.718 **beats** the model's 0.714 |
| PyPI starved of positives | only 99 malicious rows -> recall 0.222, npm->PyPI transfer ROC-AUC 0.523 |
| AST feature is dead | 97.4% sentinel values |

This notebook fixes all three **in the data**, then re-runs the decisive evaluations.

### The key idea: same-package matched negatives

For every malicious pair `(v[i-1], v[i])` we add `(v[i-2], v[i-1])` from the **same package**.

Same maintainer, same code base, same size regime, same popularity, same era.
The only thing that differs between the positive and its matched negative is **the compromise itself**.

This also destroys the ecosystem prior by construction: every package contributes one positive
and one negative, so `P(malicious | npm) -> 0.5` and the trivial baseline collapses to chance.

> **Runtime warning:** sections 2 and 3 download package archives from the live registries.
> Expect **60-120 minutes**. Every stage checkpoints to CSV, so if Colab disconnects,
> just re-run the cell and it resumes where it stopped.

In [ ]:
!pip install esprima -q
import os, io, re, ast, math, json, time, tarfile, zipfile, glob, shutil
from collections import Counter
import requests, numpy as np, pandas as pd

SEED = 42
NPM_URL  = "https://registry.npmjs.org/{p}"
PYPI_URL = "https://pypi.org/pypi/{p}/json"
HEADERS  = {"User-Agent": "supply-chain-research/1.0 (academic)"}
DELAY    = 0.4

def _get(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15); time.sleep(DELAY)
        return r.json() if r.status_code == 200 else None
    except requests.RequestException:
        return None

def fetch_npm_metadata(p):  return _get(NPM_URL.format(p=p))
def fetch_pypi_metadata(p): return _get(PYPI_URL.format(p=p))

def npm_versions(m):
    t = m.get("time", {})
    vs = [v for v in m.get("versions", {}) if v not in ("created","modified") and v != "0.0.1-security"]
    pr = [(v, t.get(v)) for v in vs if t.get(v)]; pr.sort(key=lambda x: x[1]); return pr

def pypi_versions(m):
    pr = []
    for v, files in m.get("releases", {}).items():
        if not files: continue
        ut = files[0].get("upload_time_iso_8601") or files[0].get("upload_time")
        if ut: pr.append((v, ut))
    pr.sort(key=lambda x: x[1]); return pr

print("registry helpers ready")

In [ ]:
NEEDED = ["features.csv", "merged_dataset_usable.csv"]
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print("missing:", missing)
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise SystemExit("Not on Colab - place the csv files beside this notebook.")
    for want in NEEDED:
        if not os.path.exists(want):
            cand = [g for g in glob.glob("*.csv") if g.startswith(want.replace(".csv",""))]
            if cand: shutil.copy(cand[0], want)

feat_old = pd.read_csv("features.csv")
usable   = pd.read_csv("merged_dataset_usable.csv")
print("old features:", feat_old.shape)
print(feat_old.groupby(["ecosystem","label"]).size())

---
## 1. Feature extraction

Identical to `supply_chain_detector.py` so the new rows stay comparable with the old ones,
**plus two repairs**:

* **`ast_nodes_after` is fixed.** The original called Python's `ast.parse` on the whole
  concatenated blob - which can never parse JavaScript, and fails on PyPI too because
  `.py`, `.cfg` and `.toml` files were joined into one string. Now each file is parsed
  separately with the right parser (`ast` for Python, `esprima` for JS/TS) and the node
  counts are summed.
* **Normalised counts added.** Reviewers warned the raw counts may just measure package size,
  so every signal also gets a per-KB version.

In [ ]:
SUSPICIOUS = {
  "net_http": re.compile(r"\b(https?://|requests\.(get|post)|urllib|fetch\(|XMLHttpRequest|http\.request|axios)", re.I),
  "subprocess_exec": re.compile(r"\b(subprocess|child_process|os\.system|exec\(|eval\(|spawn\(|popen)", re.I),
  "env_cred_access": re.compile(r"(process\.env|os\.environ|getenv|\.aws/credentials|id_rsa|\.npmrc|\.pypirc|SSH_|API_KEY|SECRET)", re.I),
  "filesystem": re.compile(r"\b(fs\.(write|read|unlink)|open\([^)]*['\"]w|shutil|os\.remove|writeFile)", re.I),
  "encoded_blob": re.compile(r"(base64|atob|btoa|fromCharCode|eval\(.*decode|exec\(.*decode)", re.I),
  "install_hook": re.compile(r"(preinstall|postinstall|setup\.py|cmdclass|__import__)", re.I),
  "obfuscation": re.compile(r"(_0x[0-9a-f]{4,}|[A-Za-z0-9+/]{120,})", re.I),
}

def entropy(s):
    if not s: return 0.0
    n = len(s)
    return -sum((c/n)*math.log2(c/n) for c in Counter(s).values())

def fetch_tarball_url(package, ecosystem, version):
    if ecosystem == "npm":
        m = fetch_npm_metadata(package)
        try: return m["versions"][version]["dist"]["tarball"]
        except (KeyError, TypeError): return None
    m = fetch_pypi_metadata(package)
    try:
        files = m["releases"].get(version, [])
        if not files: return None
        sd = [f for f in files if f["url"].endswith((".tar.gz", ".zip"))]
        return (sd[0] if sd else files[0])["url"]
    except (KeyError, TypeError): return None

def download_bytes(url):
    if not url: return None
    try:
        r = requests.get(url, headers=HEADERS, timeout=30); time.sleep(DELAY)
        return r.content if r.status_code == 200 else None
    except requests.RequestException: return None

def extract_files(archive, ecosystem):
    """Return list of (filename, text) instead of one blob - needed for per-file AST parsing."""
    if not archive: return []
    exts = (".js",".ts",".json",".mjs",".cjs") if ecosystem == "npm" else (".py",".cfg",".toml",".txt")
    out = []
    try:
        try:
            with tarfile.open(fileobj=io.BytesIO(archive), mode="r:gz") as t:
                for mem in t.getmembers():
                    if mem.isfile() and mem.name.endswith(exts):
                        fo = t.extractfile(mem)
                        if fo: out.append((mem.name, fo.read().decode("utf-8","ignore")))
        except tarfile.ReadError:
            with zipfile.ZipFile(io.BytesIO(archive)) as z:
                for nm in z.namelist():
                    if nm.endswith(exts):
                        out.append((nm, z.read(nm).decode("utf-8","ignore")))
    except Exception:
        return []
    return out

def extract_code(archive, ecosystem):
    return "\n".join(txt for _, txt in extract_files(archive, ecosystem))

print("extraction helpers ready")

In [ ]:
import esprima

def ast_node_count(files, ecosystem):
    """FIXED: parse each file with the correct parser and sum the nodes.
    Returns (node_count, n_parsed, n_files) so parse coverage can be reported."""
    total, ok, seen = 0, 0, 0
    for name, txt in files:
        if ecosystem == "npm":
            if not name.endswith((".js",".mjs",".cjs",".ts")): continue
            seen += 1
            try:
                tree = esprima.parseScript(txt, tolerant=True)
                total += _count_es(tree.toDict()); ok += 1
            except Exception:
                try:
                    tree = esprima.parseModule(txt, tolerant=True)
                    total += _count_es(tree.toDict()); ok += 1
                except Exception:
                    pass
        else:
            if not name.endswith(".py"): continue
            seen += 1
            try:
                total += sum(1 for _ in ast.walk(ast.parse(txt))); ok += 1
            except Exception:
                pass
    return total, ok, seen

def _count_es(node):
    if isinstance(node, dict):
        return 1 + sum(_count_es(v) for k, v in node.items() if k != "type")
    if isinstance(node, list):
        return sum(_count_es(v) for v in node)
    return 0

def pcounts(code): return {k: len(rx.findall(code)) for k, rx in SUSPICIOUS.items()}

def diff_features(fb, fa):
    cb = "\n".join(t for _, t in fb); ca = "\n".join(t for _, t in fa)
    bc, ac = pcounts(cb), pcounts(ca)
    f = {}
    for k in SUSPICIOUS:
        f[f"delta_{k}"] = ac[k] - bc[k]
        f[f"after_{k}"] = ac[k]
    f["delta_loc"]  = ca.count("\n") - cb.count("\n")
    f["delta_char"] = len(ca) - len(cb)
    f["entropy_before"] = round(entropy(cb[:50000]), 3)
    f["entropy_after"]  = round(entropy(ca[:50000]), 3)
    f["delta_entropy"]  = round(f["entropy_after"] - f["entropy_before"], 3)
    lt = lambda c: max((len(t) for t in re.findall(r"\S+", c)), default=0)
    f["delta_longest_token"] = lt(ca) - lt(cb)
    return f, cb, ca

MIN_CODE_CHARS = 30

def features_for_pair(package, ecosystem, vbefore, vafter):
    fa = extract_files(download_bytes(fetch_tarball_url(package, ecosystem, vafter)), ecosystem)
    if sum(len(t) for _, t in fa) < MIN_CODE_CHARS: return None, "after_download_failed"
    fb = extract_files(download_bytes(fetch_tarball_url(package, ecosystem, vbefore)), ecosystem)
    if sum(len(t) for _, t in fb) < MIN_CODE_CHARS: return None, "before_download_failed"

    f, cb, ca = diff_features(fb, fa)
    n_after, ok_a, seen_a = ast_node_count(fa, ecosystem)
    n_before, ok_b, seen_b = ast_node_count(fb, ecosystem)
    f["ast_nodes_after"]  = n_after
    f["ast_nodes_before"] = n_before
    f["delta_ast_nodes"]  = n_after - n_before
    f["ast_parse_rate"]   = round(ok_a / seen_a, 3) if seen_a else 0.0

    kb = max(len(ca) / 1024.0, 1.0)          # normalisation guard
    f["size_kb_after"] = round(len(ca)/1024.0, 2)
    for k in SUSPICIOUS:
        f[f"norm_after_{k}"] = round(f[f"after_{k}"] / kb, 4)
    return f, "ok"

print("feature pipeline ready (AST repaired, normalised counts added)")

---
## 2. Matched negatives

For every malicious package, walk back one more step in its own version history and build
`(v[i-2], v[i-1])` as a negative.

`v[i-1]` is exactly the version the manuscript already assumes to be benign, so this adds
**no new assumption** - it only reuses the existing one.

In [ ]:
CKPT_MATCHED = "matched_negatives.csv"

# One matched negative per MALICIOUS PAIR, not per package.
# The dataset holds 976 malicious rows spread over only ~301 packages, and 140 of those
# packages carry more than one flagged version. Keying on the package alone would throw
# away two thirds of the pairing opportunities and leave the classes imbalanced again.
mal = (feat_old[feat_old.label == 1][["package","ecosystem"]]
       .drop_duplicates()
       .merge(usable[["package","ecosystem","version_before"]].drop_duplicates(),
              on=["package","ecosystem"], how="left")
       .dropna(subset=["version_before"])
       .drop_duplicates(subset=["package","ecosystem","version_before"]))
print("malicious pairs to walk back:", len(mal),
      f"(across {mal.package.nunique()} packages)")

done = set()
if os.path.exists(CKPT_MATCHED):
    prev = pd.read_csv(CKPT_MATCHED)
    done = set(zip(prev.package, prev.ecosystem, prev.version_after))
    print("resuming - already processed:", len(done))

rows, stats = [], Counter()
for n, r in enumerate(mal.itertuples(index=False), 1):
    if (r.package, r.ecosystem, r.version_before) in done: continue
    try:
        meta = fetch_npm_metadata(r.package) if r.ecosystem == "npm" else fetch_pypi_metadata(r.package)
        if not meta: stats["no_metadata"] += 1; continue
        hist = npm_versions(meta) if r.ecosystem == "npm" else pypi_versions(meta)
        vers = [v for v, _ in hist]
        if r.version_before not in vers: stats["before_not_in_history"] += 1; continue
        i = vers.index(r.version_before)
        if i < 1: stats["no_grandparent"] += 1; continue

        v_prev, v_curr = vers[i-1], vers[i]
        f, status = features_for_pair(r.package, r.ecosystem, v_prev, v_curr)
        if f is None: stats[status] += 1; continue

        f.update(package=r.package, ecosystem=r.ecosystem, label=0,
                 version_before=v_prev, version_after=v_curr,
                 version_after_date=dict(hist)[v_curr], origin="matched_negative")
        rows.append(f); stats["ok"] += 1
    except Exception as e:
        stats[f"error:{type(e).__name__}"] += 1

    if n % 10 == 0:
        if rows:
            pd.DataFrame(rows).to_csv(CKPT_MATCHED, mode="a", header=not os.path.exists(CKPT_MATCHED),
                                      index=False)
            rows = []
        print(f"  {n}/{len(mal)}  ok={stats['ok']}  {dict(stats)}", flush=True)

if rows:
    pd.DataFrame(rows).to_csv(CKPT_MATCHED, mode="a", header=not os.path.exists(CKPT_MATCHED), index=False)

matched = pd.read_csv(CKPT_MATCHED).drop_duplicates(subset=["package","ecosystem","version_after"])
print("\nmatched negatives built:", len(matched))
print(matched.ecosystem.value_counts())
print("outcome breakdown:", dict(stats))

---
## 3. PyPI expansion

99 malicious PyPI rows is the root cause of the 0.222 recall and the failed transfer.
This pulls additional confirmed PyPI incidents from the **OpenSSF malicious-packages**
repository (already reference 39 in the manuscript) and builds version pairs for the
packages that are still retrievable.

In [ ]:
import subprocess
if not os.path.exists("malicious-packages"):
    subprocess.run(["git","clone","--depth","1","-q",
                    "https://github.com/ossf/malicious-packages.git"], check=False)

paths = glob.glob("malicious-packages/osv/malicious/pypi/**/*.json", recursive=True)
print("PyPI advisory files found:", len(paths))

adv = []
for p in paths:
    try:
        d = json.load(open(p))
    except Exception:
        continue
    for a in d.get("affected", []):
        nm = a.get("package", {}).get("name")
        if not nm: continue
        vs = a.get("versions", []) or []
        adv.append({"package": nm,
                    "versions": vs,
                    "published": d.get("published", "")})

adv = pd.DataFrame(adv).drop_duplicates(subset=["package"])
known = set(feat_old[feat_old.ecosystem == "pypi"].package)
new_pkgs = adv[~adv.package.isin(known)]
print("PyPI packages already in the dataset:", len(known))
print("NEW candidate PyPI packages:", len(new_pkgs))

In [ ]:
CKPT_PYPI = "pypi_expansion.csv"
MAX_NEW = 800          # raise if you have time; each package costs ~2 downloads

done = set()
if os.path.exists(CKPT_PYPI):
    done = set(pd.read_csv(CKPT_PYPI).package)
    print("resuming - already processed:", len(done))

rows, stats = [], Counter()
todo = new_pkgs.head(MAX_NEW)
for n, r in enumerate(todo.itertuples(index=False), 1):
    if r.package in done: continue
    try:
        meta = fetch_pypi_metadata(r.package)
        if not meta: stats["removed_from_registry"] += 1; continue
        hist = pypi_versions(meta)
        vers = [v for v, _ in hist]
        flagged = [v for v in (r.versions or []) if v in vers]
        if not flagged: stats["flagged_version_gone"] += 1; continue

        v_mal = flagged[0]
        i = vers.index(v_mal)
        if i < 1: stats["first_release_malware_excluded"] += 1; continue   # not an update compromise

        f, status = features_for_pair(r.package, "pypi", vers[i-1], v_mal)
        if f is None: stats[status] += 1; continue

        f.update(package=r.package, ecosystem="pypi", label=1,
                 version_before=vers[i-1], version_after=v_mal,
                 version_after_date=dict(hist)[v_mal], origin="pypi_expansion")
        rows.append(f); stats["ok"] += 1
    except Exception as e:
        stats[f"error:{type(e).__name__}"] += 1

    if n % 25 == 0:
        if rows:
            pd.DataFrame(rows).to_csv(CKPT_PYPI, mode="a", header=not os.path.exists(CKPT_PYPI), index=False)
            rows = []
        print(f"  {n}/{len(todo)}  {dict(stats)}")

if rows:
    pd.DataFrame(rows).to_csv(CKPT_PYPI, mode="a", header=not os.path.exists(CKPT_PYPI), index=False)

pypi_new = pd.read_csv(CKPT_PYPI).drop_duplicates(subset=["package","version_after"]) \
           if os.path.exists(CKPT_PYPI) else pd.DataFrame()
print("\nnew malicious PyPI pairs:", len(pypi_new))
print("outcome breakdown:", dict(stats))
print("\n>>> first-release malware excluded:", stats.get("first_release_malware_excluded", 0),
      "(this count answers Reviewer 1 comment 2)")

---
## 4. Rebuild the dataset

Old rows keep their original features; new rows carry the extra repaired columns.
Only the columns present everywhere are used for the headline model, so nothing is
silently imputed.

In [ ]:
feat_old = feat_old.assign(origin="original")
parts = [feat_old, matched]
if len(pypi_new): parts.append(pypi_new)
new = pd.concat(parts, ignore_index=True)

print("BEFORE:"); print(feat_old.groupby(["ecosystem","label"]).size(), "\n")
print("AFTER:");  print(new.groupby(["ecosystem","label"]).size())

for eco in ["npm","pypi"]:
    s = new[new.ecosystem == eco]
    print(f"\nP(malicious | {eco}) : {s.label.mean():.3f}   (was "
          f"{feat_old[feat_old.ecosystem==eco].label.mean():.3f})")

new.to_csv("features_v2.csv", index=False)
print("\nsaved features_v2.csv:", new.shape)

In [ ]:
# AST repair check
if "ast_parse_rate" in new.columns:
    sub = new[new.ast_parse_rate.notna()]
    print("mean AST parse coverage on NEW rows:")
    print(sub.groupby("ecosystem").ast_parse_rate.mean().round(3))
    print("\nold sentinel rate was 0.974 (97.4% failure)")

---
## 5. The decisive re-evaluation

Three questions, answered against the repaired dataset:

1. Does the **trivial ecosystem baseline** still beat the model?
2. Does **version context** still add value once the classes are matched?
3. Does anything **transfer** between ecosystems now?

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             confusion_matrix, average_precision_score, roc_auc_score)

RESULTS2 = {}
df = pd.read_csv("features_v2.csv")
df["eco_npm"] = (df.ecosystem == "npm").astype(int)

CAND = [c for c in df.columns if c.startswith("after_")] + ["entropy_after"]
PRED = [c for c in df.columns if c.startswith("delta_")] + ["entropy_before"]
CAND = [c for c in CAND if df[c].notna().all()]
PRED = [c for c in PRED if df[c].notna().all()]
print("candidate-only features:", len(CAND), "| predecessor features:", len(PRED))

y, groups = df.label.values, df.package.values
cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
def clf(): return HistGradientBoostingClassifier(max_iter=300, random_state=SEED)

def metrics(yy, pred, proba=None):
    tn, fp, fn, tp = confusion_matrix(yy, pred, labels=[0,1]).ravel()
    o = dict(f1=f1_score(yy,pred), precision=precision_score(yy,pred,zero_division=0),
             recall=recall_score(yy,pred), fpr=fp/(fp+tn) if fp+tn else 0.0,
             tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
    if proba is not None:
        o["pr_auc"] = average_precision_score(yy, proba); o["roc_auc"] = roc_auc_score(yy, proba)
    return {k:(round(v,4) if isinstance(v,float) else v) for k,v in o.items()}

def run(X, name, store):
    pr = cross_val_predict(clf(), X, y, cv=cv, groups=groups, method="predict_proba")[:,1]
    m = metrics(y, (pr>=.5).astype(int), pr)
    print(f"{name:38s} F1={m['f1']:.3f}  Rec={m['recall']:.3f}  "
          f"Prec={m['precision']:.3f}  FPR={m['fpr']:.3f}  PR-AUC={m.get('pr_auc',0):.3f}")
    RESULTS2.setdefault(store, {})[name] = m
    return m

In [ ]:
# Q1 - does the trivial baseline still win?
m_eco = metrics(y, df.eco_npm.values)
print(f"{'ECOSYSTEM-ONLY baseline':38s} F1={m_eco['f1']:.3f}  "
      f"Prec={m_eco['precision']:.3f}  FPR={m_eco['fpr']:.3f}")
print("   (Notebook 1: F1 = 0.718 and it BEAT the model)\n")

m_full = run(df[CAND+PRED], "full model (no eco flag)", "Q1")
RESULTS2["Q1"]["ecosystem_only_baseline"] = m_eco
print(f"\n>>> model - baseline = {m_full['f1'] - m_eco['f1']:+.4f} F1")

In [ ]:
# Q2 - is version context still worth the second download?
rng = np.random.RandomState(SEED)
X_full = df[CAND+PRED]
X_shuf = X_full.copy()
perm = rng.permutation(len(df))
for c in PRED: X_shuf[c] = X_shuf[c].values[perm]

m_cand = run(df[CAND], "candidate-only",          "Q2")
m_fullb= run(X_full,   "full (cand + predecessor)","Q2")
m_shuf = run(X_shuf,   "full w/ SHUFFLED pred",   "Q2")
print(f"\n>>> value of the correct predecessor = {m_fullb['f1'] - m_shuf['f1']:+.4f} F1")
print(f">>> Notebook 1 gave +0.0256 on the confounded data")

In [ ]:
# Q3 - transfer, both directions
FE = CAND + PRED
for src, tgt in [("npm","pypi"), ("pypi","npm")]:
    tr, te = df[df.ecosystem==src], df[df.ecosystem==tgt]
    if tr.label.nunique() < 2 or te.label.nunique() < 2:
        print(f"train {src} -> test {tgt}: single class, skipped"); continue
    mdl = clf().fit(tr[FE], tr.label)
    pr = mdl.predict_proba(te[FE])[:,1]
    m = metrics(te.label, (pr>=.5).astype(int), pr)
    print(f"train {src:5s} -> test {tgt:5s}  F1={m['f1']:.3f}  ROC-AUC={m['roc_auc']:.3f}")
    RESULTS2.setdefault("Q3",{})[f"{src}_to_{tgt}"] = m
print("\n>>> Notebook 1: npm->pypi ROC-AUC 0.523 (chance), pypi->npm 0.655")

In [ ]:
# per-ecosystem from the SAME unified model
pr = cross_val_predict(clf(), df[FE+["eco_npm"]], y, cv=cv, groups=groups, method="predict_proba")[:,1]
pred = (pr>=.5).astype(int)
RESULTS2["per_ecosystem"] = {}
for eco in ["npm","pypi"]:
    msk = (df.ecosystem==eco).values
    if msk.sum() and len(set(y[msk]))>1:
        m = metrics(y[msk], pred[msk], pr[msk])
        print(f"{eco:6s} F1={m['f1']:.3f}  Rec={m['recall']:.3f}  FPR={m['fpr']:.3f}")
        RESULTS2["per_ecosystem"][eco] = m

RESULTS2["dataset"] = {"n": len(df),
    "by_eco_label": {f"{a}_{b}": int(c) for (a,b),c in df.groupby(["ecosystem","label"]).size().items()},
    "prior_npm": round(float(df[df.ecosystem=='npm'].label.mean()),4),
    "prior_pypi": round(float(df[df.ecosystem=='pypi'].label.mean()),4)}

with open("revision_results_v2.json","w") as f: json.dump(RESULTS2, f, indent=2, default=str)
print("\nsaved revision_results_v2.json")
try:
    from google.colab import files
    files.download("revision_results_v2.json"); files.download("features_v2.csv")
except Exception: pass

---
## How to read the output

| Outcome | Meaning |
|---|---|
| ecosystem baseline drops to ~0.5 and the model clearly beats it | **the confound is gone** - the strongest possible answer to R1-3 |
| correct predecessor still beats shuffled | version context is real on clean data - the title survives |
| PyPI recall rises well above 0.222 | the expansion worked; per-ecosystem reporting becomes defensible |
| transfer still fails | report it honestly as a finding: malicious signals are ecosystem-specific |

Send me `revision_results_v2.json` and `features_v2.csv`.

**If sections 2-3 fail on the network**, run them again - they resume from the checkpoint files.
If a registry blocks you, raise `DELAY` to 1.0 and restart.